# Задание 1.
Реализуйте базовый класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

### Этап 1. Реализация базового класса Checkingсса Account
Класс должен быть инициализирован с параметрами:

● account_holder (str) — имя владельца счёта;

● balance (float, по умолчанию 0) — начальный баланс счёта, не может быть отрицательным.

### Атрибуты:
● _account_counter — приватный атрибут для хранения количества созданных счетов. Отсчет начинается с 1000;

● holder — хранит имя владельца;Checking

● account_number — хранит номер счёта;

● _balance — приватный атрибут для хранения текущего баланса;

● operations_history — список или другая структура для хранения истории операций.

### Важно:
Каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь, кортеж или класс. Минимальный набор данных для операции: тип операции (‘deposit’ или ‘withdraw’), сумма, дата и время операции, текущий баланс после операции, статус (‘success’ или ‘fail’).

### Этап 2. Реализация методов

init(self, account_holder, balance=0) — конструктор. Обратите внимание, что в конструкторе должен автоматически формироваться номер счёта в формате ‘ACC-XXXX’, где XXXX — порядковый номер счёта;

deposit(self, amount) — метод для пополнения счёта:

● принимает сумму (должна быть положительной), попытка положить отрицательную сумму, должна вызывать исключение;

● в случае успеха обновляет баланс и добавляет запись в историю операций.

withdraw(self, amount) — метод для снятия средств:

● принимает сумму (должна быть положительной);

● проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом ‘fail’ все равно фиксируется в истории;

● в случае успеха обновляет баланс и добавляет запись в историю.

get_balance(self) — метод, который возвращает текущий баланс.

get_history(self) — метод, который возвращает историю операций.

### Важно:
Продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

In [ ]:
from datetime import datetime
from typing import Any


class Account:
    accounts_number_order: int = 1

    def __init__(self, account_holder: str, balance: float = 0):
        self.holder: str = account_holder
        self.account_number: str = "ACC-" + str(Account.accounts_number_order).zfill(4)
        self.__operations_history: dict = {}
        self.__account_counter = 1000
        self._balance: float = 0

        self.deposit(balance)

        Account.accounts_number_order += 1

    def get_balance(self) -> float:
        return self._balance

    def get_operations_history(self) -> str:
        """
        Получение истории операций

        :return: str
        """
        result = ''

        for date, operations in self.__operations_history.items():
            if len(operations) == 0:
                continue

            result += f"{date}:\n"

            for operation in operations:
                result += "   " + " | ".join(f"{k}: {v}" for k, v in operation.items()) + "\n"

        return result

    def deposit(self, amount: float) -> bool:
        """
        Внесение суммы на счет

        :param amount:
        :raises ValueError:
        :return: True
        """
        status = amount >= 0

        if status:
            self._balance += amount

        self.log_new_operation('deposit', amount, status)

        if not status:
            raise ValueError('Сумма должна быть не отрицательной!')

        return status

    def withdraw(self, amount: float) -> bool:
        """
        Снятие денег со счета

        :param amount:
        :raises ValueError:
        :return:
        """
        error: str | False = False

        if amount < 0:
            error = 'Сумма должна быть положительной!'

        elif amount > self.get_balance():
            error = 'Сумма должна быть меньше текущего баланса!'

        else:
            self._balance -= amount

        self.log_new_operation('withdraw', amount, not bool(error))

        if error:
            raise ValueError(error)

        return True

    def log_new_operation(self, operation_type: str, data: Any, status: bool) -> None:
        """
        Сохранение новой операции

        :param operation_type:
        :param data:
        :param status:
        :return:
        """
        current_date = datetime.now()
        date = current_date.strftime("%Y-%m-%d")
        time = current_date.strftime("%H:%M:%S")

        self.__operations_history.setdefault(date, []).append({
            'time': time,
            'operation_type': operation_type,
            'status': 'success' if status else 'fail',
            'balance': self.get_balance(),
            'data': data,
        })


a = Account('test', 1000)

a.deposit(100)
a.deposit(400)
a.withdraw(0)

print(a.get_operations_history())